In [1]:
import os
import pandas as pd

# 1. Criação da Estrutura de Pastas
def inicializar_projeto():
    pastas = ['data/raw', 'data/processed', 'src', 'notebooks']
    for pasta in pastas:
        os.makedirs(pasta, exist_ok=True)
    print("✅ Estrutura de pastas verificada/criada.")

inicializar_projeto()

# 2. Séries Históricas (2014 a 2025) - 12 anos de dados
anos = list(range(2014, 2026))

# --- DIMENSÃO INFLAÇÃO (Fonte: IBGE / SIDRA) ---
dim_inflacao = pd.DataFrame({
    "sk_inflacao": range(1, 13),
    "ano": anos,
    "ipca_geral_acumulado_ano": [6.41, 10.67, 6.29, 2.95, 3.75, 4.31, 4.52, 10.06, 5.79, 4.62, 4.62, 3.80],
    "ipca_alimentos_acumulado_ano": [8.03, 12.03, 8.62, -1.87, 4.04, 6.37, 14.09, 7.94, 11.64, 1.03, 7.82, 4.10]
})

# --- DIMENSÃO POLÍTICA TRIBUTÁRIA (Fonte: Tesouro Nacional / Receita Federal) ---
# A carga tributária bruta e o peso dos impostos sobre consumo (ICMS, IPI, PIS/Cofins, ISS)
dim_tributaria = pd.DataFrame({
    "sk_tributo": range(1, 13),
    "ano": anos,
    "carga_tributaria_pib_pct": [32.42, 32.12, 31.75, 31.83, 32.51, 32.03, 31.15, 33.05, 33.07, 32.44, 32.40, 32.50],
    "peso_impostos_regressivos_consumo_pct": [44.1, 43.8, 44.5, 44.2, 45.1, 44.8, 43.9, 44.3, 44.6, 44.1, 44.2, 44.3]
})

# --- TABELA FATO: PODER DE COMPRA (Fonte: IPEADATA e PNAD/IBGE) ---
salarios_nominais = [724, 788, 880, 937, 954, 998, 1045, 1100, 1212, 1320, 1412, 1512]
indices_gini = [0.522, 0.524, 0.537, 0.538, 0.545, 0.543, 0.524, 0.544, 0.518, 0.518, 0.518, 0.519]

fato_poder_compra = pd.DataFrame({
    "ano": anos,
    "sk_inflacao": dim_inflacao["sk_inflacao"],
    "sk_tributo": dim_tributaria["sk_tributo"],
    "salario_minimo_nominal": salarios_nominais,
    "indice_gini_rendimento": indices_gini
})

# Engenharia de Recursos (Feature Engineering): 
# Calculando o Salário Mínimo Real indexado (Descontando a inflação oficial do período)
fato_poder_compra["salario_minimo_real_indexado"] = round(
    fato_poder_compra["salario_minimo_nominal"] / (1 + (dim_inflacao["ipca_geral_acumulado_ano"] / 100)), 2
)

# 3. Exportação para a Camada Processed (Star Schema Completo)
dim_inflacao.to_csv("data/processed/dim_inflacao.csv", index=False)
dim_tributaria.to_csv("data/processed/dim_tributaria.csv", index=False)
fato_poder_compra.to_csv("data/processed/fato_poder_compra.csv", index=False)

print("✅ Dados reais processados com sucesso!")
print("✅ Arquivos 'dim_inflacao.csv', 'dim_tributaria.csv' e 'fato_poder_compra.csv' salvos na pasta 'data/processed/'.")

# Exibindo uma amostra da Tabela Fato no próprio Jupyter para validação
display(fato_poder_compra.head())

✅ Estrutura de pastas verificada/criada.
✅ Dados reais processados com sucesso!
✅ Arquivos 'dim_inflacao.csv', 'dim_tributaria.csv' e 'fato_poder_compra.csv' salvos na pasta 'data/processed/'.


,ano,sk_inflacao,sk_tributo,salario_minimo_nominal,indice_gini_rendimento,salario_minimo_real_indexado
0,2014,1,1,724,0.522,680.39
1,2015,2,2,788,0.524,712.03
2,2016,3,3,880,0.537,827.92
3,2017,4,4,937,0.538,910.15
4,2018,5,5,954,0.545,919.52
